In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import when, col,sum, avg, count


In [4]:
spark = SparkSession.builder.appName("customersCSV").master("local[*]").getOrCreate()

customers_df = spark.read.option("header", "true").option("inferSchema","true").csv("DataFarmersMarket/customer.csv")

customers_df.printSchema()
customers_df.show(5)

root
 |-- customer_id: integer (nullable = true)
 |-- customer_first_name: string (nullable = true)
 |-- customer_last_name: string (nullable = true)
 |-- customer_zip: integer (nullable = true)

+-----------+-------------------+------------------+------------+
|customer_id|customer_first_name|customer_last_name|customer_zip|
+-----------+-------------------+------------------+------------+
|          1|               Jane|            Connor|       22801|
|          2|             Manuel|              Diaz|       22821|
|          3|                Bob|            Wilson|       22821|
|          4|             Deanna|        Washington|       22801|
|          5|            Abigail|            Harris|       22801|
+-----------+-------------------+------------------+------------+
only showing top 5 rows



In [8]:
# Add new columns using existing columns
churned_df = customers_df.withColumn("is_churn", when(col("customer_id")%2==0,0)
                                    .when(col("customer_id")%2!=0,1)
                                    .otherwise(-1))
churned_df.show()

+-----------+-------------------+------------------+------------+--------+
|customer_id|customer_first_name|customer_last_name|customer_zip|is_churn|
+-----------+-------------------+------------------+------------+--------+
|          1|               Jane|            Connor|       22801|       1|
|          2|             Manuel|              Diaz|       22821|       0|
|          3|                Bob|            Wilson|       22821|       1|
|          4|             Deanna|        Washington|       22801|       0|
|          5|            Abigail|            Harris|       22801|       1|
|          6|              Betty|           Bullard|       22801|       0|
|          7|            Jessica|           Armenta|       22821|       1|
|          8|              Norma|        Valenzuela|       22821|       0|
|          9|              Janet|            Forbes|       22801|       1|
|         10|            Russell|           Edwards|       22801|       0|
|         11|            

In [10]:
# Identify NULLS
customers_df.filter(col("customer_first_name").isNull()).show()

+-----------+-------------------+------------------+------------+
|customer_id|customer_first_name|customer_last_name|customer_zip|
+-----------+-------------------+------------------+------------+
+-----------+-------------------+------------------+------------+



In [11]:
# Replacing NULL values
customers_df.na.fill({"customer_first_name":"UNKNOWN"}).show()

+-----------+-------------------+------------------+------------+
|customer_id|customer_first_name|customer_last_name|customer_zip|
+-----------+-------------------+------------------+------------+
|          1|               Jane|            Connor|       22801|
|          2|             Manuel|              Diaz|       22821|
|          3|                Bob|            Wilson|       22821|
|          4|             Deanna|        Washington|       22801|
|          5|            Abigail|            Harris|       22801|
|          6|              Betty|           Bullard|       22801|
|          7|            Jessica|           Armenta|       22821|
|          8|              Norma|        Valenzuela|       22821|
|          9|              Janet|            Forbes|       22801|
|         10|            Russell|           Edwards|       22801|
|         11|            Richard|           Paulson|       22801|
|         12|               Jack|              Wise|       22821|
|         

In [12]:
#  drow rows with NULL values
# customers_df.na.drop().show() #to drop all columns
customers_df.na.drop(subset=["customer_first_name"]).show()

+-----------+-------------------+------------------+------------+
|customer_id|customer_first_name|customer_last_name|customer_zip|
+-----------+-------------------+------------------+------------+
|          1|               Jane|            Connor|       22801|
|          2|             Manuel|              Diaz|       22821|
|          3|                Bob|            Wilson|       22821|
|          4|             Deanna|        Washington|       22801|
|          5|            Abigail|            Harris|       22801|
|          6|              Betty|           Bullard|       22801|
|          7|            Jessica|           Armenta|       22821|
|          8|              Norma|        Valenzuela|       22821|
|          9|              Janet|            Forbes|       22801|
|         10|            Russell|           Edwards|       22801|
|         11|            Richard|           Paulson|       22801|
|         12|               Jack|              Wise|       22821|
|         

In [17]:
# groupby 
churned_df.groupby('customer_zip').agg(sum("is_churn").alias("total_active"),count("*").alias("total_churn")).show()

+------------+------------+-----------+
|customer_zip|total_active|total_churn|
+------------+------------+-----------+
|       22802|           2|          4|
|      122067|           1|          1|
|       22801|           8|         15|
|       22821|           3|          7|
|      110006|           0|          1|
+------------+------------+-----------+



In [19]:
product_df = spark.read.option("header", "true").option("inferSchema","true").csv("DataFarmersMarket/product.csv")
product_df.show()

+----------+--------------------+--------------+-------------------+----------------+
|product_id|        product_name|  product_size|product_category_id|product_qty_type|
+----------+--------------------+--------------+-------------------+----------------+
|         1|Habanero Peppers ...|        medium|                  1|             lbs|
|         2|Jalapeno Peppers ...|         small|                  1|             lbs|
|         3|Poblano Peppers -...|         large|                  1|            unit|
|         4|Banana Peppers - Jar|          8 oz|                  3|            unit|
|         5|   Whole Wheat Bread|       1.5 lbs|                  3|            unit|
|         6| Cut Zinnias Bouquet|        medium|                  5|            unit|
|         7|           Apple Pie|            10|                  3|            unit|
|         8|          Cherry Pie|            10|                  3|            unit|
|         9|      Sweet Potatoes|        medium|      

In [20]:
prod_cat_df = spark.read.option("header", "true").option("inferSchema","true").csv("DataFarmersMarket/product_category.csv")
prod_cat_df.show()

+-------------------+---------------------+
|product_category_id|product_category_name|
+-------------------+---------------------+
|                  1| Fresh Fruits & Ve...|
|                  2| Packaged Pantry G...|
|                  3| Packaged Prepared...|
|                  4| Freshly Prepared ...|
|                  5|     Plants & Flowers|
|                  6| Eggs & Meat (Fres...|
|                  7|  Non-Edible Products|
+-------------------+---------------------+



In [21]:
product_df.join(prod_cat_df,on='product_category_id',how='left').show()

+-------------------+----------+--------------------+--------------+----------------+---------------------+
|product_category_id|product_id|        product_name|  product_size|product_qty_type|product_category_name|
+-------------------+----------+--------------------+--------------+----------------+---------------------+
|                  1|         1|Habanero Peppers ...|        medium|             lbs| Fresh Fruits & Ve...|
|                  1|         2|Jalapeno Peppers ...|         small|             lbs| Fresh Fruits & Ve...|
|                  1|         3|Poblano Peppers -...|         large|            unit| Fresh Fruits & Ve...|
|                  3|         4|Banana Peppers - Jar|          8 oz|            unit| Packaged Prepared...|
|                  3|         5|   Whole Wheat Bread|       1.5 lbs|            unit| Packaged Prepared...|
|                  5|         6| Cut Zinnias Bouquet|        medium|            unit|     Plants & Flowers|
|                  3|       

In [22]:
product_df.join(prod_cat_df,on='product_category_id',how='right').show()

+-------------------+----------+--------------------+--------------+----------------+---------------------+
|product_category_id|product_id|        product_name|  product_size|product_qty_type|product_category_name|
+-------------------+----------+--------------------+--------------+----------------+---------------------+
|                  1|        22|       Roma Tomatoes|        medium|             lbs| Fresh Fruits & Ve...|
|                  1|        21|Organic Cherry To...|          pint|            unit| Fresh Fruits & Ve...|
|                  1|        18|   Carrots - Organic|         bunch|            unit| Fresh Fruits & Ve...|
|                  1|        17|             Carrots|sold by weight|             lbs| Fresh Fruits & Ve...|
|                  1|        16|          Sweet Corn|           Ear|            unit| Fresh Fruits & Ve...|
|                  1|        15|Red Potatoes - Small|              |            Null| Fresh Fruits & Ve...|
|                  1|       

In [26]:
# rename a column
prod_cat_df.withColumnRenamed('product_category_name','prod_cat').show()

+-------------------+--------------------+
|product_category_id|            prod_cat|
+-------------------+--------------------+
|                  1|Fresh Fruits & Ve...|
|                  2|Packaged Pantry G...|
|                  3|Packaged Prepared...|
|                  4|Freshly Prepared ...|
|                  5|    Plants & Flowers|
|                  6|Eggs & Meat (Fres...|
|                  7| Non-Edible Products|
+-------------------+--------------------+



In [27]:
# join only with specific columns
product_df[['product_category_id','product_id']].join(prod_cat_df,on='product_category_id',how='inner').show()

+-------------------+----------+---------------------+
|product_category_id|product_id|product_category_name|
+-------------------+----------+---------------------+
|                  1|        22| Fresh Fruits & Ve...|
|                  1|        21| Fresh Fruits & Ve...|
|                  1|        18| Fresh Fruits & Ve...|
|                  1|        17| Fresh Fruits & Ve...|
|                  1|        16| Fresh Fruits & Ve...|
|                  1|        15| Fresh Fruits & Ve...|
|                  1|        14| Fresh Fruits & Ve...|
|                  1|        13| Fresh Fruits & Ve...|
|                  1|        12| Fresh Fruits & Ve...|
|                  1|         9| Fresh Fruits & Ve...|
|                  1|         3| Fresh Fruits & Ve...|
|                  1|         2| Fresh Fruits & Ve...|
|                  1|         1| Fresh Fruits & Ve...|
|                  2|        23| Packaged Pantry G...|
|                  3|         8| Packaged Prepared...|
|         